# LangChain Hands-on 13 — Agents with create_agent
**Trainer:** Ajeetkumar

**What we will practise**
- Chain vs Agent
- `create_agent` (LangChain v1)
- Agent with tools — see every step
- Streaming agent steps
- Agent memory (checkpointer + thread_id)
- Structured response from an agent
- Real-life agents: travel, banking, research

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq langchain-community langgraph wikipedia ddgs

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

# Chain vs Agent
| Chain | Agent |
|---|---|
| fixed steps decided by **you** | LLM **decides** next step |
| prompt → llm → parser | think → call tool → observe → repeat |
| predictable | flexible |

**Agent = LLM + Tools + Loop** (ReAct: Reason + Act)

```
User ─▶ [Model] ─tool_calls?─ yes ─▶ [Tools] ─┐
           ▲                                  │
           └──────── results ─────────────────┘
           no ─▶ Final answer
```

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather of an Indian city."""
    data = {"pune": "28°C, cloudy", "goa": "30°C, sunny", "manali": "6°C, snow"}
    return data.get(city.lower(), "25°C, clear")

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between INR, USD and EUR."""
    rate_to_inr = {"INR": 1, "USD": 84, "EUR": 91}
    value = amount * rate_to_inr[from_currency.upper()] / rate_to_inr[to_currency.upper()]
    return f"{amount} {from_currency} = {round(value, 2)} {to_currency}"

@tool
def calculator(expression: str) -> str:
    """Solve a maths expression like '25*4+10'."""
    return str(eval(expression, {"__builtins__": {}}))

In [ ]:
tools = [get_weather, convert_currency, calculator]

## 1. Create your first agent

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful travel assistant. Use tools when needed. Keep answers short.",
)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "I am going to Goa with 500 USD. How much is that in INR and how is the weather?"}]})

In [ ]:
print(result["messages"][-1].content)

## 2. See every step the agent took

In [ ]:
for m in result["messages"]:
    m.pretty_print()

## 3. Model as a string (no need to create llm object)

In [ ]:
agent2 = create_agent("groq:openai/gpt-oss-20b", tools=tools)
agent2.invoke({"messages": [{"role": "user", "content": "What is 45*12 + 300?"}]})["messages"][-1].content

## 4. Stream the agent steps live

In [ ]:
for chunk in agent.stream({"messages": [{"role": "user", "content": "Weather in Manali and 2000 INR in EUR?"}]}, stream_mode="updates"):
    for step, data in chunk.items():
        last = data["messages"][-1]
        print(f"[{step}]", last.tool_calls if getattr(last, "tool_calls", None) else last.content)

## 5. Agent with memory — checkpointer + thread_id

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
memory_agent = create_agent(llm, tools, system_prompt="You are a travel assistant.", checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "ajeet-1"}}

In [ ]:
memory_agent.invoke({"messages": [{"role": "user", "content": "My budget is 800 EUR and I am going to Pune."}]}, config)["messages"][-1].content

In [ ]:
memory_agent.invoke({"messages": [{"role": "user", "content": "Convert my budget to INR and tell me the weather there."}]}, config)["messages"][-1].content

In [ ]:
# new thread = fresh memory
memory_agent.invoke({"messages": [{"role": "user", "content": "What is my budget?"}]}, {"configurable": {"thread_id": "new-user"}})["messages"][-1].content

## 6. Structured response from an agent

In [ ]:
from pydantic import BaseModel

class TripPlan(BaseModel):
    city: str
    weather: str
    budget_inr: float
    packing_tips: list[str]

In [ ]:
plan_agent = create_agent(llm, tools, response_format=TripPlan)
result = plan_agent.invoke({"messages": [{"role": "user", "content": "Plan for Manali with 300 USD"}]})
result["structured_response"]

## 7. Research agent with real tools

In [ ]:
from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper

In [ ]:
research_agent = create_agent(
    llm,
    tools=[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1500)), DuckDuckGoSearchRun()],
    system_prompt="You are a research assistant. Use Wikipedia for facts and web search for latest news. Give sources.",
)

In [ ]:
r = research_agent.invoke({"messages": [{"role": "user", "content": "Who founded ISRO and what is its latest mission?"}]})
print(r["messages"][-1].content)

## 8. Banking support agent (RAG-style tool + action tool)

In [ ]:
POLICY = {"lost card": "Report within 3 days for zero liability.", "upi limit": "Rs 25,000 for new payee in first 24 hours."}

@tool
def search_policy(topic: str) -> str:
    """Search bank policies. Topics: 'lost card', 'upi limit'."""
    return POLICY.get(topic.lower(), "No policy found.")

@tool
def block_card(last4: str) -> str:
    """Block the customer's card using the last 4 digits."""
    return f"Card ending {last4} is BLOCKED. New card in 5-7 days."

In [ ]:
bank_agent = create_agent(llm, [search_policy, block_card],
                          system_prompt="You are NovaBank support. Never ask for full card number, OTP or PIN.",
                          checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "cust-9"}}

In [ ]:
bank_agent.invoke({"messages": [{"role": "user", "content": "I lost my debit card yesterday, what is the rule?"}]}, cfg)["messages"][-1].content

In [ ]:
bank_agent.invoke({"messages": [{"role": "user", "content": "Block it, last 4 digits 4321"}]}, cfg)["messages"][-1].content

## 9. See the agent graph

In [ ]:
print(agent.get_graph().draw_mermaid())

# create_agent arguments
| Argument | Use |
|---|---|
| `model` | llm object or `"provider:model"` string |
| `tools` | list of tools |
| `system_prompt` | agent's role & rules |
| `checkpointer` | memory per `thread_id` |
| `response_format` | Pydantic schema → `result["structured_response"]` |
| `middleware` | guardrails, approvals, summary (next notebook) |

> Old ways (`initialize_agent`, `AgentExecutor`, `create_react_agent`) are replaced by **`create_agent`** in LangChain v1.

## 🧪 Practice
1. Build a "Student helper" agent with calculator + Wikipedia tools.
2. Add a `get_flight_price(source, destination)` tool to the travel agent.
3. Make an agent return `Invoice(customer, items, total)` as structured output.